# Custom surface site analysis

Edit **Cell 2 only**, then **Run All**.


In [165]:

# Imports (DO NOT EDIT)
import sys
sys.path.append(".")

from ase.io import read, write, Trajectory
from acat.settings import CustomSurface
from acat.adsorption_sites import SlabAdsorptionSites
from ase.build import surface
from ase.visualize import view

import site_analysis as sa


In [166]:
atoms = surface('Pd',(3,3,2),10)
atoms.center(vacuum=10, axis=2)

In [167]:
#visualize slab
write('pd332.xyz',atoms)

In [168]:
view(atoms, viewer='x3d')

In [ ]:

# USER INPUT (EDIT ONLY THIS CELL)
structure_file = "/home/shikim/pynta/pynta/preprocessing/custom_surfaces/pd332.xyz"
n_layers = 10
adsorbate_height = 1.0
site_bond_cutoff = 3.5


In [170]:

# Load slab
slab = read(structure_file)
surface = CustomSurface(slab, n_layers=n_layers)
nslab = len(slab)


In [171]:

# Generate symmetry-unique site geometries
cas = SlabAdsorptionSites(slab, surface, composition_effect=True)

single_geoms, single_sites_lists = sa.generate_unique_sites(
    slab,
    cas.get_sites(),
    nslab,
    site_bond_cutoff,
    adsorbate_height
)

traj = Trajectory("unique_sites.traj", "w")
for g in single_geoms:
    traj.write(g)
traj.close()


/home/shikim/miniconda3/envs/pynta_env/lib/python3.9/site-packages/acat/adsorption_sites.py:2128: UserWarning: Cannot identify site (37, 118, 157)
/home/shikim/miniconda3/envs/pynta_env/lib/python3.9/site-packages/acat/adsorption_sites.py:2128: UserWarning: Cannot identify site (77, 117, 158)
/home/shikim/miniconda3/envs/pynta_env/lib/python3.9/site-packages/acat/adsorption_sites.py:2128: UserWarning: Cannot identify site (38, 77, 117)
/home/shikim/miniconda3/envs/pynta_env/lib/python3.9/site-packages/acat/adsorption_sites.py:2128: UserWarning: Cannot identify site (37, 78, 157)


In [172]:

# Extract 3-fold site graphs
admols, threefold_geom_indices = sa.classify_threefold_sites(
    single_geoms, single_sites_lists
)


In [173]:

# Graph isomorphism clustering
iso_mat, clusters = sa.cluster_isomorphic_graphs(admols)


In [174]:

# Update 3-fold site labels
type_map = sa.update_threefold_site_labels(
    single_sites_lists,
    clusters,
    threefold_geom_indices
)


In [175]:

# Write 3-fold-only trajectory
traj3 = Trajectory("threefold_sites.traj", "w")
for i in threefold_geom_indices:
    traj3.write(single_geoms[i])
traj3.close()


In [176]:

# Pairwise strict isomorphism (PRINT)
print("Pairwise strict isomorphism:")
for i in range(len(admols)):
    for j in range(i + 1, len(admols)):
        print(f"{i} vs {j} =", iso_mat[i, j])


Pairwise strict isomorphism:
0 vs 1 = False
0 vs 2 = False
1 vs 2 = False


In [177]:

# Distinct 3-fold site types (PRINT)
print("Number of distinct 3-fold site types:", len(clusters))
for members in clusters.values():
    print("3-fold site type:", members)


Number of distinct 3-fold site types: 3
3-fold site type: [0]
3-fold site type: [1]
3-fold site type: [2]


In [178]:

# Updated 3-fold site labels (PRINT)
print("Updated 3-fold site labels per geometry:")
for geom_idx, label in type_map.items():
    print(f"Geometry {geom_idx} -> {label}")


Updated 3-fold site labels per geometry:
Geometry 3 -> 3fold1
Geometry 4 -> 3fold2
Geometry 6 -> 3fold3


In [179]:
# Site yaml file generated
print("All sites for the custom surfaces are saved in site.yaml")
sa.write_sites_yaml(single_sites_lists, clusters)


All sites for the custom surfaces are saved in site.yaml
